# KGiSL Chatbot - Data Inspection and Exploration

This notebook provides comprehensive tools for exploring and analyzing the KGiSL chatbot's knowledge base. Use this for data inspection, prototyping, and pipeline testing.

## Table of Contents
1. [Environment Setup](#Environment-Setup)
2. [Data Loading and Inspection](#Data-Loading-and-Inspection)
3. [Text Statistics and Analysis](#Text-Statistics-and-Analysis)
4. [Data Visualization](#Data-Visualization)
5. [Embedding Generation Testing](#Embedding-Generation-Testing)
6. [RAG Pipeline Prototyping](#RAG-Pipeline-Prototyping)
7. [Performance Analysis](#Performance-Analysis)

## Environment Setup

Import required libraries and set up the environment.

In [ ]:
# Standard libraries
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import re
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Text processing
from textstat import flesch_reading_ease, flesch_kincaid_grade
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Add project modules to path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

print("Environment setup complete!")
print(f"Project root: {project_root}")

In [ ]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Configuration
DATA_DIRS = {
    'raw': project_root / '1_data_scraping' / 'raw_data',
    'processed': project_root / '2_data_processing' / 'processed_data',
    'chunks': project_root / '2_data_processing' / 'processed_data' / 'chunks',
    'enhanced': project_root / '2_data_processing' / 'processed_data' / 'enhanced_chunks',
    'embeddings': project_root / '3_rag_pipeline' / 'embedded_chunks'
}

print("Data directories configured:")
for name, path in DATA_DIRS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {name}: {path}")

## Data Loading and Inspection

Load and inspect scraped data from different pipeline stages.

In [ ]:
def load_json_files(directory):
    """Load all JSON files from a directory"""
    data = []
    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return data
    
    for file_path in directory.glob('*.json'):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                content['source_file'] = file_path.name
                data.append(content)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return data

def load_jsonl_files(directory):
    """Load all JSONL files from a directory"""
    data = []
    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return data
    
    for file_path in directory.glob('*.jsonl'):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        chunk = json.loads(line)
                        chunk['source_file'] = file_path.name
                        data.append(chunk)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return data

# Load data from different stages
print("Loading data from different pipeline stages...")

raw_data = load_json_files(DATA_DIRS['raw'])
processed_data = load_json_files(DATA_DIRS['processed'])
chunks_data = load_jsonl_files(DATA_DIRS['chunks'])
enhanced_data = load_jsonl_files(DATA_DIRS['enhanced'])

print(f"\nData loaded:")
print(f"  Raw documents: {len(raw_data)}")
print(f"  Processed documents: {len(processed_data)}")
print(f"  Text chunks: {len(chunks_data)}")
print(f"  Enhanced chunks: {len(enhanced_data)}")

In [ ]:
# Inspect a sample document
if raw_data:
    sample_doc = raw_data[0]
    print("Sample raw document structure:")
    print(json.dumps({k: str(v)[:100] + "..." if len(str(v)) > 100 else v 
                     for k, v in sample_doc.items()}, indent=2))

if enhanced_data:
    sample_chunk = enhanced_data[0]
    print("\nSample enhanced chunk structure:")
    print(f"Text length: {len(sample_chunk.get('text', ''))}")
    print(f"Metadata keys: {list(sample_chunk.get('metadata', {}).keys())}")
    print(f"Categories: {sample_chunk.get('metadata', {}).get('categories', [])}")
    print(f"Importance score: {sample_chunk.get('metadata', {}).get('importance_score', 0)}")

## Text Statistics and Analysis

Analyze text characteristics, word frequencies, and content quality.

In [ ]:
def analyze_text_statistics(data, text_field='content'):
    """Analyze text statistics for a dataset"""
    if not data:
        return {}
    
    stats = {
        'total_documents': len(data),
        'word_counts': [],
        'char_counts': [],
        'sentence_counts': [],
        'readability_scores': [],
        'all_words': [],
        'domains': []
    }
    
    for doc in data:
        text = doc.get(text_field, '')
        if not text:
            continue
            
        # Basic counts
        words = word_tokenize(text.lower())
        sentences = sent_tokenize(text)
        
        stats['word_counts'].append(len(words))
        stats['char_counts'].append(len(text))
        stats['sentence_counts'].append(len(sentences))
        stats['all_words'].extend([w for w in words if w.isalpha()])
        
        # Readability
        try:
            stats['readability_scores'].append(flesch_reading_ease(text))
        except:
            pass
        
        # Extract domain from URL
        url = doc.get('url', doc.get('source_url', ''))
        if url:
            from urllib.parse import urlparse
            domain = urlparse(url).netloc
            stats['domains'].append(domain)
    
    return stats

# Analyze different datasets
raw_stats = analyze_text_statistics(raw_data)
chunk_stats = analyze_text_statistics(enhanced_data, 'text')

print("Text Statistics Summary:")
print("\nRaw Documents:")
if raw_stats.get('word_counts'):
    print(f"  Average words per document: {np.mean(raw_stats['word_counts']):.1f}")
    print(f"  Average characters per document: {np.mean(raw_stats['char_counts']):.1f}")
    print(f"  Average readability score: {np.mean(raw_stats['readability_scores']):.1f}")

print("\nText Chunks:")
if chunk_stats.get('word_counts'):
    print(f"  Average words per chunk: {np.mean(chunk_stats['word_counts']):.1f}")
    print(f"  Average characters per chunk: {np.mean(chunk_stats['char_counts']):.1f}")
    print(f"  Total unique words: {len(set(chunk_stats['all_words']))}")

In [ ]:
# Word frequency analysis
if chunk_stats.get('all_words'):
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_words = [w for w in chunk_stats['all_words'] 
                     if w not in stop_words and len(w) > 2]
    
    word_freq = Counter(filtered_words)
    
    print("\nTop 20 most frequent words:")
    for word, count in word_freq.most_common(20):
        print(f"  {word}: {count}")
    
    # Create word cloud
    if len(word_freq) > 0:
        plt.figure(figsize=(12, 6))
        wordcloud = WordCloud(width=800, height=400, 
                            background_color='white',
                            colormap='viridis').generate_from_frequencies(word_freq)
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word Cloud of Most Frequent Terms')
        plt.tight_layout()
        plt.show()

## Data Visualization

Create visualizations to understand data distribution and characteristics.

In [ ]:
# Create comprehensive visualizations
if chunk_stats.get('word_counts') and len(chunk_stats['word_counts']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Word count distribution
    axes[0, 0].hist(chunk_stats['word_counts'], bins=30, alpha=0.7, color='skyblue')
    axes[0, 0].set_title('Distribution of Word Counts per Chunk')
    axes[0, 0].set_xlabel('Word Count')
    axes[0, 0].set_ylabel('Frequency')
    
    # Character count distribution
    axes[0, 1].hist(chunk_stats['char_counts'], bins=30, alpha=0.7, color='lightcoral')
    axes[0, 1].set_title('Distribution of Character Counts per Chunk')
    axes[0, 1].set_xlabel('Character Count')
    axes[0, 1].set_ylabel('Frequency')
    
    # Sentence count distribution
    axes[1, 0].hist(chunk_stats['sentence_counts'], bins=20, alpha=0.7, color='lightgreen')
    axes[1, 0].set_title('Distribution of Sentence Counts per Chunk')
    axes[1, 0].set_xlabel('Sentence Count')
    axes[1, 0].set_ylabel('Frequency')
    
    # Readability scores
    if chunk_stats.get('readability_scores'):
        axes[1, 1].hist(chunk_stats['readability_scores'], bins=20, alpha=0.7, color='gold')
        axes[1, 1].set_title('Distribution of Readability Scores')
        axes[1, 1].set_xlabel('Flesch Reading Ease Score')
        axes[1, 1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze categories and importance scores
if enhanced_data:
    categories = []
    importance_scores = []
    
    for chunk in enhanced_data:
        metadata = chunk.get('metadata', {})
        categories.extend(metadata.get('categories', []))
        importance_scores.append(metadata.get('importance_score', 0))
    
    # Category distribution
    if categories:
        category_counts = Counter(categories)
        
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        categories_list = list(category_counts.keys())
        counts_list = list(category_counts.values())
        plt.bar(categories_list, counts_list, color='steelblue', alpha=0.7)
        plt.title('Distribution of Content Categories')
        plt.xlabel('Category')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        
        # Importance score distribution
        plt.subplot(1, 2, 2)
        plt.hist(importance_scores, bins=20, alpha=0.7, color='orange')
        plt.title('Distribution of Importance Scores')
        plt.xlabel('Importance Score')
        plt.ylabel('Frequency')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nCategory distribution:")
        for cat, count in category_counts.most_common():
            print(f"  {cat}: {count}")
        
        print(f"\nImportance score statistics:")
        print(f"  Mean: {np.mean(importance_scores):.3f}")
        print(f"  Std: {np.std(importance_scores):.3f}")
        print(f"  Min: {np.min(importance_scores):.3f}")
        print(f"  Max: {np.max(importance_scores):.3f}")

## Embedding Generation Testing

Test and analyze text embeddings for the RAG pipeline.

In [ ]:
# Test embedding generation
try:
    from sentence_transformers import SentenceTransformer
    
    # Load a lightweight model for testing
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"Loaded embedding model: all-MiniLM-L6-v2")
    print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
    
    # Test with sample texts
    if enhanced_data:
        sample_texts = [chunk['text'] for chunk in enhanced_data[:10]]
        
        print(f"\nGenerating embeddings for {len(sample_texts)} sample texts...")
        embeddings = model.encode(sample_texts)
        
        print(f"Embedding shape: {embeddings.shape}")
        print(f"Sample embedding statistics:")
        print(f"  Mean: {np.mean(embeddings):.4f}")
        print(f"  Std: {np.std(embeddings):.4f}")
        print(f"  Min: {np.min(embeddings):.4f}")
        print(f"  Max: {np.max(embeddings):.4f}")
        
        # Calculate similarity matrix
        similarity_matrix = cosine_similarity(embeddings)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(similarity_matrix, annot=True, cmap='coolwarm', center=0,
                   fmt='.2f', square=True)
        plt.title('Cosine Similarity Matrix of Sample Embeddings')
        plt.xlabel('Text Index')
        plt.ylabel('Text Index')
        plt.show()
        
except ImportError:
    print("sentence-transformers not available. Install with: pip install sentence-transformers")
except Exception as e:
    print(f"Error in embedding generation: {e}")

In [ ]:
# Analyze embedding clustering
try:
    if 'embeddings' in locals() and len(embeddings) > 5:
        # Reduce dimensionality for visualization
        pca = PCA(n_components=2)
        embeddings_2d = pca.fit_transform(embeddings)
        
        # Perform clustering
        n_clusters = min(3, len(embeddings) // 2)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(embeddings)
        
        # Visualize clusters
        plt.figure(figsize=(10, 6))
        scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                            c=clusters, cmap='viridis', alpha=0.7)
        plt.colorbar(scatter)
        plt.title('Text Embedding Clusters (PCA Visualization)')
        plt.xlabel('First Principal Component')
        plt.ylabel('Second Principal Component')
        
        # Add text labels for first few points
        for i, txt in enumerate(sample_texts[:5]):
            plt.annotate(f"{i}: {txt[:20]}...", 
                        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, alpha=0.7)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nClustering results:")
        for i, cluster in enumerate(clusters):
            print(f"  Text {i} -> Cluster {cluster}: {sample_texts[i][:50]}...")
            
except Exception as e:
    print(f"Error in clustering analysis: {e}")

## RAG Pipeline Prototyping

Test and prototype RAG pipeline components.

In [ ]:
# Simple RAG prototype
class SimpleRAG:
    def __init__(self, texts, embeddings=None):
        self.texts = texts
        self.embeddings = embeddings
        
        if embeddings is None and 'model' in locals():
            print("Generating embeddings for RAG...")
            self.embeddings = model.encode(texts)
    
    def retrieve(self, query, top_k=3):
        """Retrieve most relevant texts for a query"""
        if self.embeddings is None:
            return []
        
        # Generate query embedding
        query_embedding = model.encode([query])
        
        # Calculate similarities
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        
        # Get top-k most similar
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'text': self.texts[idx],
                'similarity': similarities[idx],
                'index': idx
            })
        
        return results

# Test RAG with sample data
if enhanced_data and 'model' in locals():
    sample_texts = [chunk['text'] for chunk in enhanced_data[:20]]
    rag = SimpleRAG(sample_texts)
    
    # Test queries
    test_queries = [
        "What services does KGiSL provide?",
        "Information about courses and training",
        "Contact details and location",
        "Career opportunities"
    ]
    
    for query in test_queries:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print(f"{'='*60}")
        
        results = rag.retrieve(query, top_k=3)
        
        for i, result in enumerate(results, 1):
            print(f"\nResult {i} (Similarity: {result['similarity']:.3f}):")
            print(f"{result['text'][:200]}...")
            print(f"-" * 40)
else:
    print("RAG testing requires sample data and embedding model")

## Performance Analysis

Analyze pipeline performance and identify optimization opportunities.

In [ ]:
import time

def performance_benchmark():
    """Benchmark different components of the pipeline"""
    results = {}
    
    if enhanced_data:
        # Text processing speed
        sample_texts = [chunk['text'] for chunk in enhanced_data[:100]]
        
        # Tokenization speed
        start_time = time.time()
        for text in sample_texts:
            tokens = word_tokenize(text)
        tokenization_time = time.time() - start_time
        results['tokenization_per_text'] = tokenization_time / len(sample_texts)
        
        # TF-IDF vectorization speed
        start_time = time.time()
        vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(sample_texts)
        tfidf_time = time.time() - start_time
        results['tfidf_total_time'] = tfidf_time
        results['tfidf_per_text'] = tfidf_time / len(sample_texts)
        
        # Embedding generation speed (if model available)
        if 'model' in locals():
            start_time = time.time()
            embeddings = model.encode(sample_texts[:10])  # Test with smaller sample
            embedding_time = time.time() - start_time
            results['embedding_per_text'] = embedding_time / 10
            
            # Similarity search speed
            query_embedding = model.encode(["test query"])
            start_time = time.time()
            similarities = cosine_similarity(query_embedding, embeddings)
            search_time = time.time() - start_time
            results['similarity_search_time'] = search_time
    
    return results

# Run performance benchmark
print("Running performance benchmark...")
perf_results = performance_benchmark()

print("\nPerformance Results:")
for metric, value in perf_results.items():
    if 'time' in metric:
        print(f"  {metric}: {value:.4f} seconds")
    else:
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Memory usage analysis
import psutil
import gc

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process()
    memory_info = process.memory_info()
    return memory_info.rss / 1024 / 1024  # MB

print("Memory Usage Analysis:")
print(f"Current memory usage: {get_memory_usage():.1f} MB")

# Estimate memory usage for different data sizes
if enhanced_data:
    avg_text_size = np.mean([len(chunk['text']) for chunk in enhanced_data[:100]])
    print(f"\nAverage text size: {avg_text_size:.1f} characters")
    
    # Estimate for different collection sizes
    sizes = [1000, 5000, 10000, 50000]
    for size in sizes:
        text_memory = size * avg_text_size / 1024 / 1024  # MB
        embedding_memory = size * 384 * 4 / 1024 / 1024  # 384-dim float32 embeddings
        total_memory = text_memory + embedding_memory
        print(f"  {size:,} chunks: ~{total_memory:.1f} MB (text: {text_memory:.1f} MB, embeddings: {embedding_memory:.1f} MB)")

# Cleanup
gc.collect()
print(f"\nMemory after cleanup: {get_memory_usage():.1f} MB")

## Summary and Recommendations

Based on the analysis above, generate recommendations for optimizing the chatbot pipeline.

In [ ]:
# Generate summary report
def generate_summary_report():
    """Generate a comprehensive summary report"""
    report = []
    report.append("# KGiSL Chatbot Data Analysis Summary")
    report.append(f"Analysis conducted on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("\n## Data Overview")
    
    if raw_data:
        report.append(f"- Raw documents: {len(raw_data)}")
    if enhanced_data:
        report.append(f"- Enhanced chunks: {len(enhanced_data)}")
        
        avg_words = np.mean([len(chunk['text'].split()) for chunk in enhanced_data])
        report.append(f"- Average words per chunk: {avg_words:.1f}")
        
        categories = [cat for chunk in enhanced_data 
                     for cat in chunk.get('metadata', {}).get('categories', [])]
        unique_categories = len(set(categories))
        report.append(f"- Unique content categories: {unique_categories}")
    
    report.append("\n## Recommendations")
    
    # Data quality recommendations
    if chunk_stats.get('word_counts'):
        avg_words = np.mean(chunk_stats['word_counts'])
        if avg_words < 50:
            report.append("- Consider increasing chunk size for better context")
        elif avg_words > 500:
            report.append("- Consider reducing chunk size for more precise retrieval")
    
    # Performance recommendations
    if 'perf_results' in locals():
        if perf_results.get('embedding_per_text', 0) > 0.1:
            report.append("- Consider using a faster embedding model for production")
        if perf_results.get('similarity_search_time', 0) > 0.01:
            report.append("- Consider using approximate search (FAISS) for large datasets")
    
    # Memory recommendations
    if enhanced_data and len(enhanced_data) > 10000:
        report.append("- Consider implementing streaming/batch processing for large datasets")
        report.append("- Use FAISS for efficient similarity search at scale")
    
    report.append("\n## Next Steps")
    report.append("1. Implement full data pipeline with all components")
    report.append("2. Set up production vector database (FAISS or ChromaDB)")
    report.append("3. Implement caching for frequently accessed embeddings")
    report.append("4. Add monitoring and logging for production deployment")
    report.append("5. Create evaluation metrics for RAG performance")
    
    return "\n".join(report)

# Generate and display summary
summary = generate_summary_report()
print(summary)

# Save summary to file
summary_path = project_root / 'notebooks' / 'analysis_summary.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"\nSummary saved to: {summary_path}")

## Export Functions

Utility functions for exporting analysis results and data.

In [ ]:
def export_analysis_data():
    """Export analysis results for further processing"""
    export_data = {
        'timestamp': datetime.now().isoformat(),
        'data_counts': {
            'raw_documents': len(raw_data) if raw_data else 0,
            'processed_documents': len(processed_data) if processed_data else 0,
            'text_chunks': len(chunks_data) if chunks_data else 0,
            'enhanced_chunks': len(enhanced_data) if enhanced_data else 0
        }
    }
    
    if chunk_stats.get('word_counts'):
        export_data['text_statistics'] = {
            'avg_words_per_chunk': float(np.mean(chunk_stats['word_counts'])),
            'avg_chars_per_chunk': float(np.mean(chunk_stats['char_counts'])),
            'total_unique_words': len(set(chunk_stats['all_words']))
        }
    
    if 'perf_results' in locals():
        export_data['performance'] = perf_results
    
    # Save to JSON
    export_path = project_root / 'notebooks' / 'analysis_results.json'
    with open(export_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"Analysis data exported to: {export_path}")
    return export_data

# Export analysis data
exported_data = export_analysis_data()
print("\nExported data summary:")
print(json.dumps(exported_data, indent=2))